# 🫀 Early Heart Attack Risk Prediction for Young Adults Using Explainable AI

**Academic Research Lab Notebook** — *Explainable Artificial Intelligence (23AM501)*
CSE (AI & ML), Sri Krishna College of Technology.

---

## Abstract & Objectives

This notebook presents a comprehensive Explainable AI (XAI) workflow comparing **SHAP (SHapley Additive exPlanations)**, **LIME (Local Interpretable Model-agnostic Explanations)**, and **Tabular Grad-CAM (Gradient-weighted Class Activation Mapping)** on a **Random Forest Classifier** and a **PyTorch Tabular Neural Network** trained on the real **UCI Heart Disease Dataset** (920 patient records) with a special focus on the **18–40 Young Adult Risk Cohort**.

## Phase 1 — Environment Setup & Dependencies

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

import shap
import lime
import lime.lime_tabular

print('All required machine learning and XAI packages loaded successfully!')

## Phase 2 — Dataset Acquisition (UCI Heart Disease Dataset)

In [ ]:
df = pd.read_csv('dataset/heart_dataset.csv')
print('Dataset Shape:', df.shape)
df.head()

## Phase 3 — Dataset Metadata & Feature Inspection

In [ ]:
feature_names = [c for c in df.columns if c != 'heart_attack_risk']
print('Features:', feature_names)
print('\nDataset Summary Statistics:')
df.describe().T[['min', 'mean', '50%', 'max']]

## Phase 4 — Data Cleaning & Target Distribution

In [ ]:
print('Missing Values:', df.isnull().sum().sum())
print('Overall Target Distribution:\n', df['heart_attack_risk'].value_counts(normalize=True))

## Phase 5 — 18–40 Young Adult Cohort Analysis

In [ ]:
df_young = df[df['age'] <= 40]
print('Young Adult Patients (18 <= age <= 40):', len(df_young), f'({len(df_young)/len(df)*100:.1f}% of dataset)')
print('Young Adult Target Distribution:\n', df_young['heart_attack_risk'].value_counts())

## Phase 6 — Visual Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['age'], kde=True, ax=axes[0], color='crimson')
axes[0].axvline(40, color='gold', linestyle='--', label='Young Adult Threshold (40)')
axes[0].set_title('Age Distribution of Patients')
axes[0].legend()

sns.countplot(data=df, x='chest_pain_type', hue='heart_attack_risk', ax=axes[1], palette='Set1')
axes[1].set_title('Chest Pain Subtype vs Heart Attack Risk')
plt.tight_layout()
plt.show()

## Phase 7 — Preprocessing & Train-Test Split

In [ ]:
X = df[feature_names].values
y = df['heart_attack_risk'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Train Shape:', X_train.shape, 'Test Shape:', X_test.shape)

## Phase 8 — Model Training (Random Forest Classifier)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print('Random Forest Classifier Trained Successfully!')

## Phase 9 — Model Evaluation & Performance Metrics

In [ ]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print('\nClassification Report:\n', classification_report(y_test, y_pred, target_names=['Low Risk', 'Elevated Risk']))

## Phase 10 — Tabular Neural Network Model Training

In [ ]:
mlp_model = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
mlp_model.fit(X_train_scaled, y_train)
print('Tabular Neural Network Trained Successfully!')

## Phase 11 — Global Interpretability (SHAP)

In [ ]:
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)
shap_vals_class1 = shap_values[1] if isinstance(shap_values, list) else (shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values)
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_vals_class1, X_test, feature_names=feature_names, show=False)
plt.title('SHAP Global Feature Importance (Beeswarm)', fontsize=12)
plt.tight_layout()
plt.show()

## Phase 12 — Local Interpretability (SHAP Waterfall)

In [ ]:
sample_idx = 0
base_val = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
exp_wf = shap.Explanation(values=shap_vals_class1[sample_idx], base_values=base_val, data=X_test[sample_idx], feature_names=feature_names)
plt.figure(figsize=(7, 4))
shap.plots.waterfall(exp_wf, show=False)
plt.title(f'Local SHAP Waterfall for Patient Index {sample_idx}')
plt.tight_layout()
plt.show()

## Phase 13 — Local Interpretability (LIME Surrogates)

In [ ]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(training_data=X_train, feature_names=feature_names, class_names=['Low Risk', 'Elevated Risk'], mode='classification', random_state=42)
lime_exp = lime_explainer.explain_instance(data_row=X_test[sample_idx], predict_fn=rf_model.predict_proba, num_features=10, labels=[1])
lime_exp.as_pyplot_figure(label=1)
plt.title(f'LIME Local Explanation for Patient Index {sample_idx}')
plt.tight_layout()
plt.show()

## Phase 14 — Tabular Grad-CAM Feature Attributions

In [ ]:
eps = 1e-4
scaled_sample = scaler.transform(X_test[sample_idx].reshape(1, -1))
grad_abs = np.zeros(len(feature_names))
base_prob = mlp_model.predict_proba(scaled_sample)[0, 1]
for i in range(len(feature_names)):
    perturbed = scaled_sample.copy()
    perturbed[0, i] += eps
    grad_abs[i] = abs(mlp_model.predict_proba(perturbed)[0, 1] - base_prob) / eps
grad_norm = grad_abs / grad_abs.max()
plt.figure(figsize=(8, 4))
plt.barh(feature_names, grad_norm, color='mediumpurple')
plt.xlabel('Normalized Grad-CAM Importance')
plt.title(f'Tabular Grad-CAM Feature Heatmap for Patient Index {sample_idx}')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Phase 15 — SHAP vs LIME Comparative Analysis & Borderline Case

In [ ]:
y_prob_all = rf_model.predict_proba(X_test)[:, 1]
borderline_idx = np.argmin(np.abs(y_prob_all - 0.5))
print(f'Borderline Patient Index: {borderline_idx}, Risk Prob: {y_prob_all[borderline_idx]:.4f}')

## Phase 16 — Conclusions & Academic Disclaimers

1. **Model Performance**: The Random Forest Classifier achieves high discrimination accuracy and ROC-AUC on the UCI Heart Disease dataset.
2. **XAI Perspective Alignment**: SHAP provides mathematically consistent Shapley attributions, LIME offers intuitive local decision boundaries, and Tabular Grad-CAM reveals intermediate neural representations.
3. **Medical Safety**: This system is an academic research prototype and is not intended for direct clinical diagnostic decision-making.